Ce notebook permet de créer un contexte avec plus d'informations étymologiques qu'on appellera "contexte étendu". Le principe est simple, nous allons étendre la colonne etym du contexte de base en remontant l'étymologie. Quand la description étymologique possède un étymon, nous allons chercher sa propre définition étymologique. Pour ce faire, nous utilisons des expressions régulières. Le processus d'extension de l'étymologie s'arrête une fois qu'aucune expression régulière ne matche avec la description étymologique.

Nous avons également extrait les déclinaisons du dernier étymon lorsqu'il en possédait.



# Imports et configuration

In [ ]:
# Uniquement pour Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uniquement pour Colab
%cd '/content/drive/MyDrive/FAC/Master/M2/Stage/Code/Gold_context'
# path Maïwenn à changer

/content/drive/MyDrive/FAC/Master/M2/Stage/Code/Gold_context


In [ ]:
! pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import json
import re
from tqdm import tqdm
import ast
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from unidecode import unidecode
import requests

#Extension du contexte

In [ ]:
df = pd.read_csv("./gold_context.csv")
df

,mot,catégorie,définition,étymologie,famille
0,lire,Verbe,['Interpréter des informations écrites sous fo...,"['Du latin lĕgĕre (« id. »), proprement «\u202...","['délire', 'entrelire', 'lisage', 'liseur', 'l..."
1,siège,Nom commun,['Meuble utilisé pour s’asseoir.'],"['Du latin sediculum (« siège, chaise, banquet...","['assiéger', 'siéger', 'télésiège', 'cul', 'de..."
2,chaise,Nom commun,"['Siège avec dossier, sans accoudoirs.']",['De chaire par assibilation dialectale du \\r...,"['chaisier', 'chaisière', 'minichaise', 'seize..."
3,fauteuil,Nom commun,"['Siège comportant des accotoirs, et un dossie...","['Du moyen français fauteuil, de l’ancien fran...","['chesterfield', 'fauteuil']"
4,oiseau,Nom commun,"['Animal vertébré théropode, à deux pattes et ...","['Du moyen français oiseau, oyseau, de l’ancie...","['antioiseau', 'oiseaulogue', 'oiselage', 'ois..."
...,...,...,...,...,...
95,coturnisme,Nom commun,['Intoxication aiguë due aux toxines sécrétées...,['Du latin coturnix « caille ».'],"['caillage', 'caille', 'caillement', 'cailler'..."
96,bibition,Nom commun,['Action de boire.'],['bas latin bibitio (« même sens »)'],"['bibition', 'boire', 'bu', 'buvable', 'buvard..."
97,thalassique,Adjectif,"['Marin, aquatique.']",['Du latin thalassicus (« couleur de la mer »).'],"['maritime', 'thalassique']"
98,thélite,Nom commun,['Inflammation du mamelon.'],"['Du grec ancien θηλή, thêlế (« mamelle ») et ...","['thélalgie', 'thélorrhagie', 'thélotisme', 'm..."


In [ ]:
def get_all_languages_pattern() :
  url = "https://fr.wiktionary.org/wiki/Wiktionnaire:Liste_des_langues"
  req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
  html_page = urlopen(req).read()
  soup = BeautifulSoup(html_page, 'html.parser')
  all_languages = []
  for x in soup.select('tbody>tr')[4:] :
    try :
      lg = x.select("td")[2]["data-sort-value"]
      all_languages.append(lg)
    except :
      pass
  all_languages_sorted = sorted(all_languages, key = lambda x : len(x), reverse=True)
  return "(" + "|".join(all_languages_sorted) + ")"


In [ ]:
lang_pattern = get_all_languages_pattern()
pattern1 = re.compile("(de|du|de la|de l'|de l’)\\s*" + lang_pattern + "\\s+(\\w+)", re.IGNORECASE)
pattern2 = re.compile("dérivé\\w* de (\\w+)", re.IGNORECASE)
pattern3 = re.compile("^\\w+ de (\\w+)", re.IGNORECASE)

In [ ]:
f = re.search( pattern1, "dérivé du grec ancien Ἀγαμέμνων, Agamemnon, le roi de l'Océan" )
print(f.group(3))

Ἀγαμέμνων


In [ ]:
class EtymExtension :

  def __init__(self, etym, lang) :
    self.etym = etym
    self.lang = lang
    self.next = None
    f1 = re.search(pattern1, etym)
    f2 = re.search(pattern2, etym)
    f3 = re.search(pattern3, etym)
    if f1 :
      self.next = f1.group(3)
      self.lang = f1.group(2)
      self.pattern = 1
    elif f2 :
      self.next = f2.group(1)
      self.pattern = 2
    elif f3 :
      self.next = f3.group(1)
      self.pattern = 3

  def __str__(self) :
    return "No next" if self.next is None else f'Next "{self.next}" ({self.lang}) found with pattern {self.pattern}'

  def __repr__(self) :
    return "No next" if self.next is None else f'Next "{self.next}" ({self.lang}) found with pattern {self.pattern}'

In [ ]:
ext = EtymExtension("Féminin de petraeus (« pierreux »).", "français")
print(ext)

Next "petraeus" (français) found with pattern 3


In [ ]:
ecrit_grc = "Petraea"
unidecode_treat = unidecode(ecrit_grc)
unidecode_treat

'Petraea'

In [ ]:
def search_in_kaikki(etymExtensions) :
  res = []
  for etymextension in etymExtensions :
    if etymextension is None or etymextension.next is None :
      res.append(None)
      continue
    if etymextension.lang == "latin" :
      word_next = unidecode(etymextension.next)
    else :
      word_next = etymextension.next

    url = f"https://kaikki.org/frwiktionary/All languages combined/meaning/{word_next[0]}/{word_next[:2]}/{word_next}.jsonl"

    r = requests.get(url)
    if r.status_code != 200 :
      res.append(None)
      continue

    entries = [json.loads(line) for line in r.text.strip().splitlines()]
    entries = [e for e in entries if e.get("lang", "").lower() == etymextension.lang.lower()]
    res.append(entries if entries else None)

  return res

In [ ]:
etyms = [EtymExtension(eval(etym)[0], "français") if isinstance(etym, str) else None for etym in df["étymologie"] ]
search_in_kaikki(etyms)

[[{'word': 'legere',
   'lang_code': 'la',
   'lang': 'Latin',
   'pos': 'verb',
   'pos_title': 'Forme de verbe',
   'senses': [{'glosses': ['Infinitif de lego.'],
     'tags': ['form-of'],
     'form_of': [{'word': 'lego'}]}],
   'sounds': [{'ipa': '\\le.ɡe.re\\'}],
   'categories': ['Formes de verbes en latin', 'latin'],
   'tags': ['form-of']}],
 [{'word': 'sediculum',
   'lang_code': 'la',
   'lang': 'Latin',
   'pos': 'noun',
   'pos_title': 'Nom commun',
   'etymology_texts': ['Dérivé de sedes (« siège »), avec le suffixe -culum, forme collatérale neutre\u202f^([1]) de sēdēcŭla.'],
   'senses': [{'glosses': ['Petit siège.'],
     'categories': ['Wiktionnaire:Exemples manquants en latin']}],
   'forms': [{'form': 'sedicula', 'tags': ['plural', 'nominative']},
    {'form': 'sedicula', 'tags': ['plural', 'vocative']},
    {'form': 'sedicula', 'tags': ['plural', 'accusative']},
    {'form': 'sediculī', 'tags': ['singular', 'genitive']},
    {'form': 'sediculōrum', 'tags': ['plural',

In [ ]:
def etym_extraction(kaikki_results) : # kaikki_results c'est ce qu'on obtient quand on fait serach_in_kaikki(etyms)
    if kaikki_results is None or len(kaikki_results) == 0 :
        return None
    etym = kaikki_results[0].get("etymology_texts")
    if not etym:
        return None
    return etym[0] # parce que etymology_texts est une liste dans kaikki, même si en général il n'y a qu'un élément dedans, bah nous on veut le premier

def get_extension_batchwise(etymologies, langs=None, acc=None, verbose=True) :
    if all( [x is None for x in etymologies] ) :
        return acc

    if langs is None :
        langs = ["français" for i in etymologies]
    if acc is None :
        acc = [ [] for i in etymologies ]

    if verbose:
        print("Level", len(acc[0]))

    etymExtensions = [None if etym is None else EtymExtension(etym, lang) for etym, lang in zip(etymologies, langs)]
    res = search_in_kaikki(etymExtensions)
    acc = [former + [new] for former, new in zip(acc, res)]

    next_etymologies = [etym_extraction(kaikki_results) for kaikki_results in res] # pour pouvoir analyser la string de l'étymologie extraite du fichier jsonl trouvé sur kaikki
    next_langs = [None if ext is None else ext.lang for ext in etymExtensions]

    return get_extension_batchwise(next_etymologies, next_langs, acc)

In [ ]:
etymologies = [eval(etym)[0] if isinstance(etym, str) else None for etym in df["étymologie"]]
result = get_extension_batchwise(etymologies)

Level 0
Level 1
Level 2
Level 3
Level 4


In [ ]:
etyms = []
declinaisons = []

for idx, entry in enumerate(result) :
  etym = []
  declinaison = []
  last_elem = None
  for elem in entry :
    #print(f"elem : {elem}")
    if elem is not None :
      last_elem = elem
      for d in elem :
        #print(f"dictionnaire : {d}")
        if "etymology_texts" in d :
          etym.append( d["etymology_texts"] )
  if last_elem is not None :
    definition = last_elem[0]["senses"][0]["glosses"][0]
    etym.append(definition)
    if "forms" in last_elem[0] :
      declinaison = last_elem[0]["forms"]
    #print(f"etym : {etym}")
  etyms.append(etym)
  declinaisons.append(declinaison)

df["étymologie étendue"] = etyms
df["déclinaison"] = declinaisons

In [ ]:
df

,mot,catégorie,définition,étymologie,famille,étymologie étendue,déclinaison
0,lire,Verbe,['Interpréter des informations écrites sous fo...,"['Du latin lĕgĕre (« id. »), proprement «\u202...","['délire', 'entrelire', 'lisage', 'liseur', 'l...",[Infinitif de lego.],[]
1,siège,Nom commun,['Meuble utilisé pour s’asseoir.'],"['Du latin sediculum (« siège, chaise, banquet...","['assiéger', 'siéger', 'télésiège', 'cul', 'de...","[[Dérivé de sedes (« siège »), avec le suffixe...",[]
2,chaise,Nom commun,"['Siège avec dossier, sans accoudoirs.']",['De chaire par assibilation dialectale du \\r...,"['chaisier', 'chaisière', 'minichaise', 'seize...","[[Du grec ancien καθέδρα, kathédra.], [Motcomp...",[]
3,fauteuil,Nom commun,"['Siège comportant des accotoirs, et un dossie...","['Du moyen français fauteuil, de l’ancien fran...","['chesterfield', 'fauteuil']",[],[]
4,oiseau,Nom commun,"['Animal vertébré théropode, à deux pattes et ...","['Du moyen français oiseau, oyseau, de l’ancie...","['antioiseau', 'oiseaulogue', 'oiselage', 'ois...",[[Du radical indo-européen commun *au̯ei- (« o...,"[{'form': 'avēs', 'tags': ['plural', 'nominati..."
...,...,...,...,...,...,...,...
95,coturnisme,Nom commun,['Intoxication aiguë due aux toxines sécrétées...,['Du latin coturnix « caille ».'],"['caillage', 'caille', 'caillement', 'cailler'...",[Caille.],"[{'form': 'coturnicēs', 'tags': ['plural', 'no..."
96,bibition,Nom commun,['Action de boire.'],['bas latin bibitio (« même sens »)'],"['bibition', 'boire', 'bu', 'buvable', 'buvard...",[],[]
97,thalassique,Adjectif,"['Marin, aquatique.']",['Du latin thalassicus (« couleur de la mer »).'],"['maritime', 'thalassique']","[[Du grec ancien θαλασσικός, thalassikós.], Ma...","[{'form': 'thalassică', 'tags': ['singular', '..."
98,thélite,Nom commun,['Inflammation du mamelon.'],"['Du grec ancien θηλή, thêlế (« mamelle ») et ...","['thélalgie', 'thélorrhagie', 'thélotisme', 'm...","[[De θάω, tháō (« sucer, téter »), apparenté a...","[{'form': 'θηλαί', 'tags': ['plural', 'nominat..."


In [ ]:
df = df[['mot', 'catégorie', 'définition', 'étymologie', 'étymologie étendue', 'déclinaison', 'famille']]

In [ ]:
df

,mot,catégorie,définition,étymologie,étymologie étendue,déclinaison,famille
0,lire,Verbe,['Interpréter des informations écrites sous fo...,"['Du latin lĕgĕre (« id. »), proprement «\u202...",[Infinitif de lego.],[],"['délire', 'entrelire', 'lisage', 'liseur', 'l..."
1,siège,Nom commun,['Meuble utilisé pour s’asseoir.'],"['Du latin sediculum (« siège, chaise, banquet...","[[Dérivé de sedes (« siège »), avec le suffixe...",[],"['assiéger', 'siéger', 'télésiège', 'cul', 'de..."
2,chaise,Nom commun,"['Siège avec dossier, sans accoudoirs.']",['De chaire par assibilation dialectale du \\r...,"[[Du grec ancien καθέδρα, kathédra.], [Motcomp...",[],"['chaisier', 'chaisière', 'minichaise', 'seize..."
3,fauteuil,Nom commun,"['Siège comportant des accotoirs, et un dossie...","['Du moyen français fauteuil, de l’ancien fran...",[],[],"['chesterfield', 'fauteuil']"
4,oiseau,Nom commun,"['Animal vertébré théropode, à deux pattes et ...","['Du moyen français oiseau, oyseau, de l’ancie...",[[Du radical indo-européen commun *au̯ei- (« o...,"[{'form': 'avēs', 'tags': ['plural', 'nominati...","['antioiseau', 'oiseaulogue', 'oiselage', 'ois..."
...,...,...,...,...,...,...,...
95,coturnisme,Nom commun,['Intoxication aiguë due aux toxines sécrétées...,['Du latin coturnix « caille ».'],[Caille.],"[{'form': 'coturnicēs', 'tags': ['plural', 'no...","['caillage', 'caille', 'caillement', 'cailler'..."
96,bibition,Nom commun,['Action de boire.'],['bas latin bibitio (« même sens »)'],[],[],"['bibition', 'boire', 'bu', 'buvable', 'buvard..."
97,thalassique,Adjectif,"['Marin, aquatique.']",['Du latin thalassicus (« couleur de la mer »).'],"[[Du grec ancien θαλασσικός, thalassikós.], Ma...","[{'form': 'thalassică', 'tags': ['singular', '...","['maritime', 'thalassique']"
98,thélite,Nom commun,['Inflammation du mamelon.'],"['Du grec ancien θηλή, thêlế (« mamelle ») et ...","[[De θάω, tháō (« sucer, téter »), apparenté a...","[{'form': 'θηλαί', 'tags': ['plural', 'nominat...","['thélalgie', 'thélorrhagie', 'thélotisme', 'm..."


In [ ]:
df.to_csv("./gold_context_extended.csv", index=False)